In [1]:
!pip install faiss-cpu sentence-transformers pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 50.1 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

zip_path = '/content/drive/MyDrive/loan.zip'
extract_to = '/content/loan_data/'

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_to)

print("Extracted files:")
for f in os.listdir(extract_to):
    print(f)

Mounted at /content/drive
Extracted files:
loan_approval_dataset.csv


In [8]:
import pandas as pd

csv_path = '/content/loan_data/loan_approval_dataset.csv'
df = pd.read_csv(csv_path)

# Clean column names AND string values
df.columns = df.columns.str.strip()
df['loan_status'] = df['loan_status'].str.strip()

print(df.shape)
print(df.columns.tolist())
print("Loan status values:", df['loan_status'].unique())
df.head()

(4269, 13)
['loan_id', 'no_of_dependents', 'education', 'self_employed', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value', 'loan_status']
Loan status values: ['Approved' 'Rejected']


,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


In [10]:
import numpy as np

approved = df[df['loan_status'] == 'Approved']
rejected = df[df['loan_status'] == 'Rejected']

stat_documents = [
    f"The average CIBIL score of approved applicants is {approved['cibil_score'].mean():.0f}, compared to {rejected['cibil_score'].mean():.0f} for rejected applicants.",
    f"Approved applicants have an average annual income of {approved['income_annum'].mean():.0f}, while rejected applicants average {rejected['income_annum'].mean():.0f}.",
    f"The average loan amount for approved applications is {approved['loan_amount'].mean():.0f} vs {rejected['loan_amount'].mean():.0f} for rejected ones.",
    f"Approved applicants have an average loan term of {approved['loan_term'].mean():.1f} months vs {rejected['loan_term'].mean():.1f} months for rejected.",
    f"Self-employed applicants make up {(approved['self_employed']=='Yes').mean()*100:.1f}% of approvals and {(rejected['self_employed']=='Yes').mean()*100:.1f}% of rejections.",
    f"Applicants with 0 dependents account for {(approved['no_of_dependents']==0).mean()*100:.1f}% of approvals.",
    f"Average residential assets for approved applicants: {approved['residential_assets_value'].mean():.0f} vs {rejected['residential_assets_value'].mean():.0f} for rejected.",
    f"Average commercial assets for approved: {approved['commercial_assets_value'].mean():.0f} vs rejected: {rejected['commercial_assets_value'].mean():.0f}.",
    f"Average luxury assets for approved: {approved['luxury_assets_value'].mean():.0f} vs rejected: {rejected['luxury_assets_value'].mean():.0f}.",
    f"Average bank assets for approved: {approved['bank_asset_value'].mean():.0f} vs rejected: {rejected['bank_asset_value'].mean():.0f}.",
    f"Approval rate in dataset: {len(approved)/len(df)*100:.1f}%. Rejection rate: {len(rejected)/len(df)*100:.1f}%.",
    f"CIBIL scores below {rejected['cibil_score'].quantile(0.75):.0f} cover 75% of rejected applicants.",
    f"CIBIL scores above {approved['cibil_score'].quantile(0.25):.0f} cover 75% of approved applicants.",
    f"Applicants with income above {approved['income_annum'].quantile(0.75):.0f} are almost always approved.",
    f"Loan amounts exceeding {rejected['loan_amount'].quantile(0.75):.0f} are frequently rejected.",
]

print(f"Generated {len(stat_documents)} statistical rule documents from dataset.")
for doc in stat_documents[:5]:
    print("-", doc)

Generated 15 statistical rule documents from dataset.
- The average CIBIL score of approved applicants is 703, compared to 429 for rejected applicants.
- Approved applicants have an average annual income of 5025904, while rejected applicants average 5113825.
- The average loan amount for approved applications is 15247252 vs 14946063 for rejected ones.
- Approved applicants have an average loan term of 10.4 months vs 11.7 months for rejected.
- Self-employed applicants make up 0.0% of approvals and 0.0% of rejections.


In [11]:
def row_to_text(row):
    """Convert a dataset row into a natural language case document."""
    status = row['loan_status']
    return (
        f"A {'self-employed' if row['self_employed'] == 'Yes' else 'salaried'} applicant "
        f"with {row['no_of_dependents']} dependents, "
        f"CIBIL score of {row['cibil_score']}, "
        f"annual income of {row['income_annum']}, "
        f"requested a loan of {row['loan_amount']} "
        f"for {row['loan_term']} months. "
        f"Residential assets: {row['residential_assets_value']}, "
        f"commercial assets: {row['commercial_assets_value']}, "
        f"bank assets: {row['bank_asset_value']}. "
        f"Loan was {status}."
    )

# Sample up to 500 cases (balanced approved/rejected) to keep index manageable
approved_sample = approved.sample(min(250, len(approved)), random_state=42)
rejected_sample = rejected.sample(min(250, len(rejected)), random_state=42)
sampled_df = pd.concat([approved_sample, rejected_sample]).reset_index(drop=True)

case_documents = [row_to_text(row) for _, row in sampled_df.iterrows()]

print(f"Generated {len(case_documents)} case documents from real dataset rows.")
print("\nExample case:")
print(case_documents[0])

Generated 500 case documents from real dataset rows.

Example case:
A salaried applicant with 4 dependents, CIBIL score of 794, annual income of 4200000, requested a loan of 15500000 for 18 months. Residential assets: 7300000, commercial assets: 700000, bank assets: 2200000. Loan was Approved.


In [12]:
knowledge_base = stat_documents + case_documents

print(f"Total knowledge base size: {len(knowledge_base)} documents")
print(f"  - Statistical rules: {len(stat_documents)}")
print(f"  - Real case evidence: {len(case_documents)}")

Total knowledge base size: 515 documents
  - Statistical rules: 15
  - Real case evidence: 500


In [13]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(knowledge_base, show_progress_bar=True, batch_size=64)
embeddings = np.array(embeddings).astype('float32')

print(f"Embeddings shape: {embeddings.shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Embeddings shape: (515, 384)


In [14]:
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

print(f"FAISS index built. Total vectors: {index.ntotal}")

FAISS index built. Total vectors: 515


In [15]:
def retrieve(query: str, top_k: int = 3) -> list:
    """
    Retrieve top_k most relevant documents for a given applicant query.
    Returns list of dicts with rank, document text, and distance score.
    """
    query_vec = model.encode([query]).astype('float32')
    distances, indices = index.search(query_vec, top_k)

    results = []
    for rank, (idx, dist) in enumerate(zip(indices[0], distances[0]), start=1):
        results.append({
            "rank": rank,
            "document": knowledge_base[idx],
            "distance": round(float(dist), 4)
        })
    return results


def applicant_to_query(row: pd.Series) -> str:
    """Convert a dataset row into a retrieval query."""
    return (
        f"Applicant with CIBIL score {row.get('cibil_score')}, "
        f"income {row.get('income_annum')}, "
        f"loan amount {row.get('loan_amount')}, "
        f"loan term {row.get('loan_term')} months, "
        f"self employed: {row.get('self_employed')}, "
        f"dependents: {row.get('no_of_dependents')}, "
        f"residential assets: {row.get('residential_assets_value')}, "
        f"bank assets: {row.get('bank_asset_value')}."
    )

In [16]:
sample = df.iloc[0]
query = applicant_to_query(sample)

print("Applicant Query:\n", query)
print(f"\nActual loan status: {sample['loan_status']}\n")
print("=" * 60)
print("🔍 Retrieved Evidence:\n")

results = retrieve(query, top_k=3)
for r in results:
    print(f"[{r['rank']}] {r['document']}")
    print(f"     Distance: {r['distance']}\n")

Applicant Query:
 Applicant with CIBIL score 778, income 9600000, loan amount 29900000, loan term 12 months, self employed:  No, dependents: 2, residential assets: 2400000, bank assets: 8000000.

Actual loan status: Approved

🔍 Retrieved Evidence:

[1] A salaried applicant with 0 dependents, CIBIL score of 877, annual income of 7500000, requested a loan of 25100000 for 14 months. Residential assets: 21000000, commercial assets: 11400000, bank assets: 10700000. Loan was Approved.
     Distance: 0.1658

[2] A salaried applicant with 0 dependents, CIBIL score of 831, annual income of 3700000, requested a loan of 12300000 for 16 months. Residential assets: 8000000, commercial assets: 2200000, bank assets: 2400000. Loan was Approved.
     Distance: 0.1676

[3] A salaried applicant with 0 dependents, CIBIL score of 786, annual income of 4700000, requested a loan of 15200000 for 10 months. Residential assets: 11100000, commercial assets: 8000000, bank assets: 3200000. Loan was Approved.
     

In [17]:
import json

faiss.write_index(index, '/content/drive/MyDrive/loan_faiss.index')

with open('/content/drive/MyDrive/knowledge_base.json', 'w') as f:
    json.dump(knowledge_base, f, indent=2)

print("✅ Saved to Google Drive:")
print("   - loan_faiss.index")
print("   - knowledge_base.json")

✅ Saved to Google Drive:
   - loan_faiss.index
   - knowledge_base.json
